In [3]:
# Extract_Enlistment_Database.py
# 5/21/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import random
import requests
import math
import urllib3



# Define the URL and the data
url = "https://httpbin.org/post"


def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp
    
def convert_date_index(event_date_raw):
    if type(event_date_raw)==str and event_date_raw.find(r"T")>0:date_obj = dt.strptime(str(event_date_raw), "%Y-%m-%dT%H:%M:%S");
    if type(event_date_raw)==str and event_date_raw.find(r"/")>0:  date_obj = dt.strptime(event_date_raw, "%m/%d/%Y")    
    if isinstance(event_date_raw, pd.Timestamp):               date_obj = dt.strptime(str(event_date_raw), "%Y-%m-%d %H:%M:%S")
    if type(event_date_raw)==str and len(event_date_raw)==11:  date_obj = dt.strptime(event_date_raw, '%b %d %Y')
    if type(event_date_raw)==str and len(event_date_raw)==6:  date_obj = dt.strptime(event_date_raw, "%d %b");date_obj = date_obj.replace(year=2024)
    event_date_index = date_obj.strftime("%Y%m%d")
    event_date =date_obj.strftime("%m/%d/%Y")
    return event_date,event_date_index
    
def convert_date_z(date): 
    index = date.find(r"T")
    date0=left(date,index)
    date0=dt.strptime(date0, "%Y-%m-%d")
    date_out=date0.strftime("%m/%d/%Y")
    return date_out

def convert_date(date): 
    date0=dt.strptime(date, "%m/%d/%Y")
    date_out=date0.strftime("%m/%d/%Y")
    return date_out

def convert_time(event_time): 
    if type(event_time)==str and event_time.find(r"T")>0 : date_obj = dt.strptime(str(event_time), "%Y-%m-%dT%H:%M:%S") 
    if isinstance(event_time, pd.Timestamp):               date_obj = dt.strptime(str(event_time), "%Y-%m-%d %H:%M:%S")  
    if type(event_time)==str and ( len(event_time)==11 or len(event_time)==7 or len(event_time)==8):\
                    date_obj = dt.strptime(event_time, '%I:%M %p')
    date_obj = date_obj + timedelta(hours=1)
    date_obj = date_obj.replace(minute=0)
    event_time_rounded = date_obj.strftime('%H:%M')
    event_time_index=date_obj.strftime('%H')  
    return event_time_rounded,event_time_index

def random_sleep(call_count): 
    global sleep_time_pause,sleep_range
    if call_count%100==0:
        sleep_time =sleep_time_pause
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds • Sleeping till: {sleep_end_date}")
    else:
        sleep_time = random.uniform(sleep_range[0],sleep_range[1])
    return sleep_time 

def get_record_count(beautifulSoupText):
    bold_tags=beautifulSoupText.find_all('b')
    number=0
    for tag in bold_tags:
        match = re.search(r'[\d,]+', tag.text)
        if match:
            number = match.group()
            number=int(number.replace(",",""))
            # print(number)
            return number
    print('### No Record Count Fount ####')
    return number



# Intercept network requests
def interceptor_get(request):
    # Change the method to POST
    global payload
    request.method = 'GET'
    request.headers['sec-ch-ua']= '"Chromium";v="136", "Google Chrome";v="136", "Not.A/Brand";v="99"'
    request.headers['sec-ch-ua-platform']= '"Windows"'
    request.headers['sec-fetch-mode']= 'navigate'
    request.headers['user-agent']= 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36'
    request.headers['Accept']= 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7'
    request.headers['Accept-language']= 'en-GB,en-US;q=0.9,en;q=0.8,ja;q=0.7'
    request.headers['cookie']= 'JSESSIONID=67ACCDBAAF4EA8173B9E5B5EBB49F053; saved_q_form=dt%23%3D%23893%23cf%23q%23%3D%23%23cf%23as_alq%23%3D%23%23cf%23as_anq%23%3D%23%23cf%23as_epq%23%3D%23%23cf%23as_woq%23%3D%23%23cf%23nfo_24994%23%3D%23V%2C8%2C1900%23cf%23op_24994%23%3D%23select0%23cf%23txt_24994%23%3D%23%23cf%23nfo_24995%23%3D%23V%2C24%2C1900%23cf%23op_24995%23%3D%23select0%23cf%23txt_24995%23%3D%23%23cf%23nfo_24996%23%3D%23V%2C2%2C1900%23cf%23cl_24996%23%3D%23%23cf%23nfo_24998%23%3D%23V%2C3%2C1900%23cf%23cl_24998%23%3D%23%23cf%23nfo_24997%23%3D%23V%2C4%2C1900%23cf%23cl_24997%23%3D%23%23cf%23nfo_24993%23%3D%23V%2C2%2C1900%23cf%23op_24993%23%3D%23select0%23cf%23txt_24993%23%3D%2342%23cf%23nfo_24981%23%3D%23V%2C1%2C1900%23cf%23cl_24981%23%3D%23%23cf%23nfo_24983%23%3D%23V%2C2%2C1900%23cf%23op_24983%23%3D%23select0%23cf%23txt_24983%23%3D%23; session_id=BPNZYGVALRJXWI; AWSALB=N4zhzzVL0+57uwD3lVIgSZ43bfPrt6baBi8C+6EXOtbZBO35VNVHIbLqK+tUCaRGUwlaSq8fUQSbtve7Nt3O48Bs/x2L5XmqqK3+9kdeLg0iSJaBVXSr+3k4oOwz; AWSALBCORS=N4zhzzVL0+57uwD3lVIgSZ43bfPrt6baBi8C+6EXOtbZBO35VNVHIbLqK+tUCaRGUwlaSq8fUQSbtve7Nt3O48Bs/x2L5XmqqK3+9kdeLg0iSJaBVXSr+3k4oOwz; BIGipServerbIXmoWTiygQUiepePoFgTA=!I+TuyBNPEgzKHR+QGdu8VyoScYjoMbId3VRXbLfUi9zthu4GP0ihHK2qu/sNQ8N2a2gnBX6CONYYwd8=; JSESSIONID=9F301C543785D585C2BD5C4FF9019DEF; AWSALB=UPfrus684C06njfnEosL697PZ9q0dWpE2Hd/bfUdA8BPqSv1bHGVjlOdr/i+0P83KW3zRH5hIPYvXHuWe+cRCppl7kEoFWDWocLvHvxqyeRaxOLV9r4CxMOMsEA8; AWSALBCORS=UPfrus684C06njfnEosL697PZ9q0dWpE2Hd/bfUdA8BPqSv1bHGVjlOdr/i+0P83KW3zRH5hIPYvXHuWe+cRCppl7kEoFWDWocLvHvxqyeRaxOLV9r4CxMOMsEA8; BIGipServerbIXmoWTiygQUiepePoFgTA=!GkdStryeGIbRD0WQGdu8VyoScYjoMTwcHY3jvvH2amLtpj3mf2LgRv3sFXrlBBGgBtd184IjmO6HNlI='
    request.headers['referer']='https://aad.archives.gov/aad/display-partial-records.jsp?dt=893&sc=24994%2C24995%2C24996%2C24998%2C24997%2C24993%2C24981%2C24983&cat=WR26&tf=F&bc=%2Csl%2Cfd&q=&as_alq=&as_anq=&as_epq=&as_woq=&nfo_24994=V%2C8%2C1900&op_24994=0&txt_24994=&nfo_24995=V%2C24%2C1900&op_24995=0&txt_24995=&nfo_24996=V%2C2%2C1900&cl_24996=&nfo_24998=V%2C3%2C1900&cl_24998=&nfo_24997=V%2C4%2C1900&cl_24997=&nfo_24993=V%2C2%2C1900&op_24993=0&txt_24993=&nfo_24981=V%2C1%2C1900&cl_24981=&nfo_24983=V%2C2%2C1900&op_24983=0&txt_24983=&mtch=3030407&pg=3'
    request.headers['upgrade-insecure-requests']= '1'
    request.headers['Host']= 'aad.archives.gov'
    request.headers['pragma']= 'no-cache'
    request.headers['accept-encoding']= 'gzip, deflate, br, zstd'

def extract_name(name):
    name_clean=name.replace('##','#').replace('##','#').replace('##','#').replace('##','#').replace('##','#')
    name_clean=name_clean.replace("MC#","Mc")
    name_clean=name_clean.replace("VAN#","VAN_")
    name_parts=name_clean.split("#")
    last_name=name_parts[0]
    first_name=name_parts[1]
    if len(name_parts)>=4:
        middle_initial=name_parts[2]
    else:
        middle_initial="nmn"
    return last_name,first_name,middle_initial

#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
start_time=datetime.datetime.now()
event_time=start_time

process_name='Extract Enlistment Database'
print('***',process_name,' Process Started')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"


fileInput="Correlated_Soldiers_Name_List_X"
fileOutput='Enlistment_Database'

if append==1:fileIn='Correlated_Soldiers_Name_List_X'

# fileInHtml=DirMain+"DataFiles\\"+fileInput+".html"
fileIn=DirMain+"Data\\"+fileInput+".csv"
fileOut=DirMain+"OutputData\\ABMC\\"+fileOutput+'_'+timestamp+".csv"
fileOut0=DirMain+"OutputData\\ABMC\\"+fileOutput+".csv"
fileOutInt=DirMain+"OutputData\\ABMC\\"+fileOutput+"_Interim.csv"

field_list=['serial_number','last_name','first_name','middle_initial','home_state','home_county','enlistment_location','enlistment_date','birth_year']

names_list=pd.read_csv(fileIn)
soldiers_df = pd.DataFrame([])

# with open(fileInHtml, "r", encoding="utf-8") as f:Archival_Databases_html = f.read()
# beautifulSoupText = BeautifulSoup(Archival_Databases_html, "html.parser")

options = webdriver.ChromeOptions()
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)
driver.request_interceptor = interceptor_get   

# url0 = "https://aad.archives.gov/aad/display-partial-records.jsp?dt=893&sc=24994%2C24995%2C24996%2C24998%2C24997%2C24993%2C24981%2C24983&cat=WR26&tf=F&bc=%2Csl%2Cfd&q=&as_alq=&as_anq=&as_epq=&as_woq=&nfo_24994=V%2C8%2C1900&op_24994=0&txt_24994=&nfo_24995=V%2C24%2C1900&op_24995=1&txt_24995="
# url1="&nfo_24996=V%2C2%2C1900&cl_24996=&nfo_24998=V%2C3%2C1900&cl_24998=&nfo_24997=V%2C4%2C1900&cl_24997=&nfo_24993=V%2C2%2C1900&op_24993=0&txt_24993=42&nfo_24981=V%2C1%2C1900&cl_24981=&nfo_24983=V%2C2%2C1900&op_24983=0&txt_24983=&mtch=2174&rpp=50&pg="

print(f" ***** Start {process_name} Processing ****")

url0 = "https://aad.archives.gov/aad/display-partial-records.jsp?dt=893&sc=24994%2C24995%2C24996%2C24998%2C24997%2C24993%2C24981%2C24983&cat=WR26&tf=F&bc=%2Csl%2Cfd&q=&as_alq=&as_anq=&as_epq=&as_woq=&nfo_24994=V%2C8%2C1900&op_24994=0&txt_24994=&nfo_24995=V%2C24%2C1900&op_24995=2&txt_24995="
url1="&nfo_24996=V%2C2%2C1900&cl_24996=&nfo_24998=V%2C3%2C1900&cl_24998=&nfo_24997=V%2C4%2C1900&cl_24997=&nfo_24993=V%2C2%2C1900&op_24993=0&txt_24993=&nfo_24981=V%2C1%2C1900&cl_24981=&nfo_24983=V%2C2%2C1900&op_24983=0&txt_24983=&mtch=5856&rpp=200&pg="
url1a="&rpp=200"

sleep_range=[5,15]
sleep_time_pause=1*60
get_counter=0
name_count=0

total_name_count=len(names_list)
for name in names_list['last_name']:
    record_count=0
    total_record_count=20000
    max_pages=1000
    name_count+=1
    page=1
    print('Name:',name)
    while record_count<total_record_count:
        url=url0+name.upper()+url1+str(page)+url1a
        # print(url,record_count,total_record_count)
        # input('')
        driver.get(url)

        response_text = driver.page_source
        url_source=response_text   
        beautifulSoupText = BeautifulSoup(url_source, "html.parser")  
        if beautifulSoupText.text.find('No records found')>0 or beautifulSoupText.text.find('Error Occurred')>0:break
        total_record_count=get_record_count(beautifulSoupText)
        data_rows = beautifulSoupText.find_all('tr')
        # print('total_record_count:',total_record_count,'\n')
        data_rows = [row for row in data_rows if not row.find('td', class_='sort_column')]
        
        for row in data_rows:
            row_text=row.text.strip()           
            if not "#" in row_text:continue
            record=row.find_all('td')
            serial_number=record[1].text.strip()
            current_name=record[2].text.strip()
            last_name,first_name,middle_initial= extract_name(current_name)
            state=record[3].text.strip()
            county=record[4].text.strip()
            enlistment_location=record[5].text.strip()
            enlistment_date=record[6].text.strip()
            birth_year=record[8].text.strip()
            soldiers_line_df=pd.DataFrame([serial_number,last_name,first_name,middle_initial,state,county,enlistment_location,enlistment_date,birth_year]).T
            soldiers_line_df.columns=field_list      
            soldiers_df = soldiers_df._append(soldiers_line_df, ignore_index = True) 
            total_records_processed=len(soldiers_df)
            record_count+=1
            # print(record_count,total_record_count)
            # input('')
        page+=1
        print(f"    --- {process_name:<20} • Records Processed- Name / Total: {record_count}/{total_record_count} • Name: {name}  • Name count: {name_count}/{total_name_count}  ")  
        get_counter+=1
        sleep_time=random_sleep(get_counter)       
        time.sleep(sleep_time) 
        soldiers_df.to_csv(fileOutInt,mode="w", index=False,header=True)

driver.quit() 

end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=processing_time.total_seconds()
soldiers_df.to_csv(fileOut0,mode="w", index=False,header=True)
soldiers_df.to_csv(fileOut,mode="w", index=False,header=True) 
    
end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=processing_time.total_seconds()

print("\n")
print(' **** Saved To: ',fileOut0)
# print(f"**** Process Complete *** Extract StubHub Events *** , Time: {end_time},Process Time: {int(processing_time_seconds)} Seconds ****") 



*** Extract Enlistment Database  Process Started
 ***** Start Extract Enlistment Database Processing ****
Name: Warren
    --- Extract Enlistment Database • Records Processed- Name / Total: 200/5856 • Name: Warren  • Name count: 1/643  
    --- Extract Enlistment Database • Records Processed- Name / Total: 400/5856 • Name: Warren  • Name count: 1/643  
    --- Extract Enlistment Database • Records Processed- Name / Total: 600/5856 • Name: Warren  • Name count: 1/643  
    --- Extract Enlistment Database • Records Processed- Name / Total: 800/5856 • Name: Warren  • Name count: 1/643  
    --- Extract Enlistment Database • Records Processed- Name / Total: 1000/5856 • Name: Warren  • Name count: 1/643  
    --- Extract Enlistment Database • Records Processed- Name / Total: 1200/5856 • Name: Warren  • Name count: 1/643  
    --- Extract Enlistment Database • Records Processed- Name / Total: 1400/5856 • Name: Warren  • Name count: 1/643  
    --- Extract Enlistment Database • Records Proces

In [ ]:
beautifulSoupText.text

In [ ]:
soldiers_df

In [ ]:
        match = re.search(r'[\d,]+', tag.text)
        if match:
            number = match.group()
            print(number)

In [ ]:
name='aaron'
page=1
url0 = "https://aad.archives.gov/aad/display-partial-records.jsp?dt=893&sc=24994%2C24995%2C24996%2C24998%2C24997%2C24993%2C24981%2C24983&cat=WR26&tf=F&bc=%2Csl%2Cfd&q=&as_alq=&as_anq=&as_epq=&as_woq=&nfo_24994=V%2C8%2C1900&op_24994=0&txt_24994=&nfo_24995=V%2C24%2C1900&op_24995=1&txt_24995="
url1="%23%23%23%23%23%23%23%23%23%23%23%23%23%23%23%23%23%23%23&nfo_24996=V%2C2%2C1900&cl_24996=&nfo_24998=V%2C3%2C1900&cl_24998=&nfo_24997=V%2C4%2C1900&cl_24997=&nfo_24993=V%2C2%2C1900&op_24993=0&txt_24993=42&nfo_24981=V%2C1%2C1900&cl_24981=&nfo_24983=V%2C2%2C1900&op_24983=0&txt_24983=&mtch=2174&rpp=50&pg="
url1a="&rpp=50"

url=url0+name.upper()+url1+str(page)+url1a

